In [1]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

In [3]:
pdf = pd.read_csv("data/output/actual/pdf_goals_agg.csv")
total_games = 26
overall_games_played = 5
remaining_games = total_games - overall_games_played

In [4]:

pdf["goals_per_game"] = pdf["goals"] / overall_games_played

In [5]:
pdf

,player_id,goals,full_name,team,goals_per_game
0,WSL_0055,5,Océane Hurtré,Birmingham City,1.0
1,WSL_0231,5,Khadija Shaw,Manchester City,1.0
2,WSL_0308,5,Victoria Pelova,Tottenham Hotspur,1.0
3,WSL_0233,4,Lauren Hemp,Manchester City,0.8
4,WSL_0205,3,Daniëlle van de Donk,London City Lionesses,0.6
5,WSL_0069,3,Kiko Seike,Brighton and Hove Albion,0.6
6,WSL_0320,3,Viviane Asseyi,West Ham,0.6
7,WSL_0349,3,Alexia Putellas,London City Lionesses,0.6
8,WSL_0289,3,Olivia Holdt,Tottenham Hotspur,0.6
9,WSL_0230,2,Mary Fowler,Manchester City,0.4


In [6]:


pdf["prob_0_goals"] = poisson.pmf(0, pdf["goals_per_game"])
pdf["prob_1_goal"] = poisson.pmf(1, pdf["goals_per_game"])
pdf["prob_2_goals"] = poisson.pmf(2, pdf["goals_per_game"])
pdf["prob_3plus_goals"] = poisson.sf(2, pdf["goals_per_game"])

In [7]:
rng = np.random.default_rng(seed=42)

In [8]:
num_sims = 10000

In [9]:
pdf_temp = pd.DataFrame()

In [10]:
import pandas as pd

simulations = []

for a in range(num_sims):
    pdf_temp = pdf.copy()

    for i in range(remaining_games):
        pdf_temp[f"simulated_goals_week_{i + 1}"] = rng.poisson(
            lam=pdf_temp["goals_per_game"]
        )

    week_cols = [
        f"simulated_goals_week_{i + 1}"
        for i in range(remaining_games)
    ]

    pdf_temp["simulated_goals"] = pdf_temp[week_cols].sum(axis=1)
    pdf_temp["overall_goals"] = (
        pdf_temp["goals"] + pdf_temp["simulated_goals"]
    )
    pdf_temp["sim_number"] = a + 1

    simulations.append(pdf_temp)

pdf_results = pd.concat(simulations, ignore_index=True)

In [11]:
pdf = pdf_results#.query("sim_number == 1")

In [12]:
pdf_pics = pd.read_csv("data/intermediate/actual/player_pics.csv")

In [13]:
a = pdf_pics[["name", "full_name"]].merge(pdf[["full_name", "overall_goals", "sim_number"]], on="full_name")

In [14]:
b = pd.read_csv("data/output/actual/player_pics.csv")[["full_name", "num_picks"]]

In [15]:
c = a.merge(b, on="full_name")

In [16]:
c["points"] = c["overall_goals"] / c["num_picks"]

In [17]:
d = c.groupby(["name", "sim_number"])[["overall_goals", "points"]].sum().sort_values("points", ascending=False).reset_index()

In [18]:
d["rank"] = (
    d.groupby("sim_number")["points"]
    .rank(method="min", ascending=False)
    .astype(int)
)

In [26]:
rank_probabilities = (
    d.groupby(["name", "rank"])
    .size()
    .div(d["sim_number"].nunique())
    .mul(100)
    .unstack(fill_value=0)
    .rename_axis(columns="rank")
    .reset_index()
)

In [31]:
rank_probabilities.sort_values(1, ascending=False).head(20)#.to_csv("data/output/actual/rank_probabilities.csv", index=False)

rank,name,1,2,3,4,5,6,7,8,9,10,11,12,13
9,Samuel Lewis,50.39,23.67,14.71,8.18,1.86,0.82,0.28,0.08,0.01,0.00,0.00,0.00,0.00
3,Holly Tidball,20.77,29.14,26.74,17.50,3.81,1.56,0.40,0.08,0.00,0.00,0.00,0.00,0.00
12,Tom Knuckey,20.00,29.15,28.23,17.17,3.71,1.17,0.38,0.19,0.00,0.00,0.00,0.00,0.00
7,Megan Luxford,9.45,17.75,24.18,31.84,10.44,4.00,1.66,0.55,0.11,0.01,0.00,0.01,0.00
11,Sophie Lewis,0.25,0.86,2.82,9.11,22.48,19.93,18.76,17.24,7.08,0.57,0.80,0.10,0.00
4,Ian Lewis,0.03,0.36,2.34,10.37,35.54,31.34,15.67,3.84,0.46,0.03,0.02,0.00,0.00
8,Paddy Smith,0.02,0.20,0.85,4.73,16.62,19.78,24.09,24.04,9.36,0.19,0.11,0.01,0.00
10,Simon Watson,0.01,0.01,0.11,0.83,2.72,4.70,8.22,15.77,39.55,1.15,22.72,4.12,0.09
0,Alice Teigh,0.00,0.01,0.01,0.47,3.81,16.90,30.24,34.02,12.76,0.79,0.91,0.08,0.00
1,Amy Kanold-Madla,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.05,13.71,86.24


In [19]:
# e = d.query("rank == 1")

In [20]:
# e.groupby("name")[["sim_number"]].count().reset_index().sort_values("sim_number", ascending=False).rename(columns={"sim_number": "num_wins"})

In [21]:
# num_2 = d.query("rank == 2")
# num_3 = d.query("rank == 3")

In [22]:
# num_2.groupby("name")[["sim_number"]].count().reset_index().sort_values("sim_number", ascending=False).rename(columns={"sim_number": "num_wins"})

In [23]:
# num_3.groupby("name")[["sim_number"]].count().reset_index().sort_values("sim_number", ascending=False).rename(columns={"sim_number": "num_wins"})